In [1]:
import os, time, random, zipfile
from calendar import monthrange
from concurrent.futures import ThreadPoolExecutor, as_completed
import cdsapi

import xarray as xr
import numpy as np

NETID = "k16v981"
BASE  = f"/home/{NETID}/my_work/data/era5"

# ✅ Save all daily files directly into 'utci_daily/'
OUT_UTCI = os.path.join(BASE, "utci_daily")
os.makedirs(OUT_UTCI, exist_ok=True)

ARABIA_AREA = [35.0, 30.0, 5.0, 65.0]  # N, W, S, E

# --- runtime knobs from environment (with safe defaults) ---
YEAR        = int(os.environ.get("YEAR", 2004))   # sbatch sets this
MAX_WORKERS = int(os.environ.get("MAX_WORKERS", 2))
MAX_RETRIES = int(os.environ.get("MAX_RETRIES", 5))
BASE_SLEEP  = float(os.environ.get("BASE_SLEEP", 2.0))
JITTER_MAX  = float(os.environ.get("JITTER_MAX", 2.0))

# polite pacing helper
def _sleep_paced(mult=1.0):
    time.sleep(BASE_SLEEP * mult + random.uniform(0, JITTER_MAX))

# CDS client (needs ~/.cdsapirc set up)
c = cdsapi.Client()

# -------------------------
# ERA5-HEAT (UTCI) daily
# Dataset: 'derived-utci-historical'
# NOTE: Sub-area selection is *not supported* by CDS for this dataset; requests return global archives.
# We'll pull month-by-month to keep file sizes manageable and clip later in xarray/cdo.
# -------------------------
def dl_utci_day(year: int, month: int, day: int):
    """
    Download one day of ERA5-HEAT UTCI (hourly) as a zip, extract first .nc.
    Returns (y, m, d, status).
    NOTE: Sub-area selection is typically ignored; clip later.
    """
    ymd = f"{year}_{month:02d}_{day:02d}"
    # final daily-maximum file we want to keep
    daily_target = os.path.join(OUT_UTCI, f"era5heat_utci_max_{ymd}.nc")
    # temporary hourly file (will be deleted)
    hourly_target = os.path.join(OUT_UTCI, f"era5heat_utci_hourly_{ymd}.nc")
    zip_target    = os.path.join(OUT_UTCI, f"era5heat_utci_{year}{month:02d}{day:02d}.zip")

    # if daily-max already exists, skip this day
    if os.path.exists(daily_target):
        print(f"✅ daily max exists: {ymd}")
        return (year, month, day, "exists")

    req = {
        "product_type": "consolidated_dataset",
        "version": "1_1",
        "variable": "universal_thermal_climate_index",
        "year": [f"{year:04d}"],
        "month": [f"{month:02d}"],
        "day": [f"{day:02d}"],
        "time": [f"{h:02d}:00" for h in range(24)],  # be explicit: all hours
        "format": "zip",
        "area": ARABIA_AREA,  # [N, W, S, E]
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"📥 UTCI {ymd} … (attempt {attempt})")
            if os.path.exists(zip_target):
                try: os.remove(zip_target)
                except OSError: pass

            c.retrieve("derived-utci-historical", req, zip_target)

            # 1) Extract the first .nc in the zip to a temporary hourly file
            with zipfile.ZipFile(zip_target, "r") as zf:
                nc_members = [n for n in zf.namelist() if n.lower().endswith(".nc")]
                if not nc_members:
                    raise RuntimeError(f"No .nc inside {zip_target}")
                nc_name = nc_members[0]
                with zf.open(nc_name) as src, open(hourly_target, "wb") as dst:
                    dst.write(src.read())

            # 2) Open hourly file with xarray and compute daily max over "time"
            ds = xr.open_dataset(hourly_target)
            # assume the variable is the first data_var (utci)
            vname = list(ds.data_vars)[0]
            utci = ds[vname]

            # daily max over time dimension
            utci_max = utci.max(dim="time", keep_attrs=True)

            # add a single time coord (midnight of this date, for example)
            utci_max = utci_max.expand_dims(
                time=[np.datetime64(f"{year:04d}-{month:02d}-{day:02d}")]
            )
            utci_max.name = "utci_max"

            # 3) Save daily max to NetCDF (compressed)
            comp = {"zlib": True, "complevel": 4}
            utci_max.to_netcdf(daily_target, encoding={"utci_max": comp})

            ds.close()

            # 4) Clean up: remove zip + hourly .nc
            try:
                if os.path.exists(zip_target):
                    os.remove(zip_target)
            except OSError:
                pass

            try:
                if os.path.exists(hourly_target):
                    os.remove(hourly_target)
            except OSError:
                pass

            print(f"✅ daily max saved: {ymd}")
            _sleep_paced()  # gentle spacing
            return (year, month, day, "success")

        except Exception as e:
            print(f"❌ {ymd} failed (attempt {attempt}): {e}")
            _sleep_paced(mult=attempt)  # exponential-ish backoff

    # final cleanup if all retries failed
    try:
        if os.path.exists(zip_target):
            os.remove(zip_target)
    except OSError:
        pass
    return (year, month, day, "failed")



def run_utci_year_pool(year: int, max_workers: int = MAX_WORKERS):
    jobs = []
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for m in range(1, 13):
            for d in range(1, monthrange(year, m)[1] + 1):
                jobs.append(ex.submit(dl_utci_day, year, m, d))
        for fut in as_completed(jobs):
            results.append(fut.result())
    # retry any failures once (serial)
    failed = [(y, m, d) for (y, m, d, s) in results if s == "failed"]
    if failed:
        print("\n🔁 Retrying failed days once more (serial)…")
        for y, m, d in failed:
            dl_utci_day(y, m, d)
    return results


In [2]:
if __name__ == "__main__":
    print(f"=== Starting UTCI download for YEAR={YEAR} ===")
    
    run_utci_year_pool(YEAR)              
    

=== Starting UTCI download for YEAR=2023 ===
📥 UTCI 2023_01_01 … (attempt 1)
📥 UTCI 2023_01_02 … (attempt 1)


2025-11-25 03:26:25,454 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.


2025-11-25 03:26:25,456 INFO Request ID is 0fe17dcd-d608-48fe-8d9b-af30907f39f9


2025-11-25 03:26:25,639 INFO status has been updated to accepted


2025-11-25 03:26:26,049 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.


...

HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 1:
  #000: H5A.c line 528 in H5Aopen_by_name(): can't open attribute
    major: Attribute
    minor: Can't open object
  #001: H5VLcallback.c line 1091 in H5VL_attr_open(): attribute open failed
    major: Virtual Object Layer
    minor: Can't open object
  #002: H5VLcallback.c line 1058 in H5VL__attr_open(): attribute open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: H5VLnative_attr.c line 130 in H5VL__native_attr_open(): can't open attribute
    major: Attribute
    minor: Can't open object
  #004: H5Aint.c line 545 in H5A__open_by_name(): unable to load attribute info from object header
    major: Attribute
    minor: Unable to initialize object
  #005: H5Oattribute.c line 494 in H5O__attr_open_by_name(): can't locate attribute: '_QuantizeBitGroomNumberOfSignificantDigits'
    major: Attribute
    minor: Object not found
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 1:
  #000: H5A.c line 528 in H5Ao

✅ daily max saved: 2023_12_30


9de78ce6b3da2072255534bdbc4d0bd4.zip:   0%|          | 0.00/892k [00:00<?, ?B/s]

/tmp/ipykernel_851875/1124867153.py:96: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  utci_max = utci_max.expand_dims(


✅ daily max saved: 2023_12_31
